# ViZDoom My Way Home: PPO + Random Network Distillation (RND) in PyTorch

In this notebook, we're going to train a PPO agent with Random Network Distillation to beat the My Way Home scenario in ViZDoom, using only visual observations from the game!

My Way Home is a maze. Doomguy is dropped into a random room, and a green armor vest is always in the same one. The rooms are different colors, so the screen is basically the map. The official writeup of the scenario is [here](https://vizdoom.farama.org/environments/default/#my-way-home). The catch is that the reward is very sparse: nothing happens until he stumbles onto the vest, so an agent that's just wandering gets almost no signal to learn from.

RND, from Burda et al.'s [Exploration by Random Network Distillation](https://arxiv.org/abs/1810.12894), is a simple way to reward novelty. A **target** network is initialized randomly and never trained. A **predictor** network is trained to match the target's output on every frame the agent sees. On frames like the ones it has trained on, the predictor gets close. On new frames, it doesn't. So its error is a measure of how unfamiliar a frame is, and it gets paid out as a bonus reward. Unlike curiosity methods built on a forward model, there's nothing to predict about the future, so it can't be fooled by things that are just random.

This one is deliberately plain. It's the stock maze with the stock rewards and the stock actions, with no reward shaping. Everything is written in plain PyTorch, and each piece is small and lives in its own cell: the encoder, the actor-critic, RND, the rollout buffer, and the PPO update.

It also tries to use the original Doom WADs if it can find them.

The notebook was designed to run in place in the repo, however it also supports running on Google Colab, with the option of using Google Drive to save artifacts.

# Setup

Install the pinned dependencies first.

In [1]:
import sys

# Update the required system packages when running on Colab.
if "google.colab" in sys.modules:
    !apt-get update -qq
    !apt-get install -y -qq libopenal1 libopenal-dev > /dev/null

# Install the Python packages used throughout the notebook.
%pip install -q vizdoom==1.3.1 tqdm==4.67.3 gymnasium==1.3.0 imageio==2.38.0 imageio-ffmpeg==0.6.0 opencv-python==5.0.0.93 scipy==1.17.1 numpy==2.4.6 pandas==3.0.5 torch==2.13.0 matplotlib==3.11.1

Note: you may need to restart the kernel to use updated packages.


# Imports

In [2]:
import json
import os
import shutil
import subprocess
import sys
from datetime import datetime
from pathlib import Path

import gymnasium
import imageio.v2 as imageio
import imageio_ffmpeg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from gymnasium.spaces import Box
from gymnasium.wrappers import FrameStackObservation, ResizeObservation, TransformObservation
from IPython.display import Image, display
from scipy.io import wavfile
from torch.distributions import Categorical
from tqdm.auto import tqdm


import vizdoom as vzd
from vizdoom import gymnasium_wrapper  # registers the Vizdoom* gymnasium envs
from vizdoom.gymnasium_wrapper.base_gymnasium_env import VizdoomEnv

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Experiment Setup
The observation setup matches the other notebooks. The agent acts every 4 frames and sees the last 4 frames stacked at 84x84, which gives it a sense of motion. My Way Home has 5 buttons, and we allow only one at a time, so the actions are turn left, turn right, walk forward, step left, step right, and do nothing. We run 16 environments side by side like the other PPO notebooks, because every update learns from a fresh batch collected across all of them.

Evaluation happens during training. At the end of every generation, the current policy plays `EVAL_EPISODES` episodes on a separate environment, and each one is saved as a `.lmp` demo while it plays. Those demos are tiny, so we keep all of them. After training, `RECORD_EPISODES` demos from the best generation are turned into videos. Set `RECORD_CHECKPOINTS` to `"all"` to convert that many from every generation too, although that takes a long while.

In [3]:
ENV_NAME = "VizdoomMyWayHome-v1"
NUM_ENVS = 16
FRAME_SKIP = 4          # act every 4 frames
STACK_SIZE = 4          # last 4 frames, so the agent can see motion
OBS_SIZE = (84, 84)
MAX_BUTTONS_PRESSED = 1 # one button at a time

EVAL_EPISODES = 5       # recorded evaluation episodes at the end of every generation
RECORD_EPISODES = 3     # demos per generation converted to video
RECORD_CHECKPOINTS = "best"  # or "all"

All the path logic lives in the next few cells, so the rest of the notebook doesn't need to care whether it's running on Colab or locally.

On Colab, `USE_DRIVE` keeps the run in Google Drive so it survives the session. Otherwise, everything stays in the notebook's folder.

In [4]:
USE_DRIVE = True # Colab only: keep the experiment (and every output) in Google Drive
EXPERIMENT = "my-way-home-ppo-rnd-pytorch"

if "google.colab" in sys.modules and USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    experiment_dir = Path("/content/drive/MyDrive/rl-epsilon/vizdoom-agilerl-experiments") / EXPERIMENT
elif "google.colab" in sys.modules:
    experiment_dir = Path("/content") / EXPERIMENT  # lost when the Colab session ends
else:
    experiment_dir = Path.cwd()  # local: the notebook's own folder
print("experiment:", experiment_dir)

experiment: /Users/coldstart/Desktop/rl-epsilon/vizdoom-agilerl-experiments/my-way-home-ppo-rnd-pytorch


ViZDoom ships with Freedoom. But if `DOOM2.WAD` is in the `WADS` folder, we use that instead. Always better to use the original assets in my opinion! The scenario is the stock maze, `my_way_home.cfg`, which ships with ViZDoom.

In [5]:
USE_DOOM2_WAD = True  # use DOOM2.WAD if found, otherwise ViZDoom's bundled Freedoom
STOCK_SCENARIO = "my_way_home.cfg"  # bundled with ViZDoom

scenario_cfg = str(Path(vzd.scenarios_path) / STOCK_SCENARIO)
SCENARIO_NAME = Path(scenario_cfg).stem

wad = experiment_dir.parent / "WADS" / "DOOM2.WAD"
DOOM_WAD = str(wad) if USE_DOOM2_WAD and wad.is_file() else None
IWAD_NAME = Path(DOOM_WAD).name if DOOM_WAD else "freedoom2"
print("scenario:", scenario_cfg)
print("IWAD:", IWAD_NAME)

scenario: /opt/miniconda3/lib/python3.13/site-packages/vizdoom/scenarios/my_way_home.cfg
IWAD: DOOM2.WAD


Each run gets its own timestamped folder. The best checkpoint, charts, and videos are copied to `artifacts`.

In [6]:
run_dir = experiment_dir / "training-runs" / f"run-{datetime.now():%Y%m%d_%H%M%S}"
checkpoint_dir = run_dir / "checkpoints"
recording_dir = run_dir / "recordings"
artifacts_dir = experiment_dir / "artifacts"
model_dir = artifacts_dir / "model"
video_dir = artifacts_dir / "video"
figure_dir = artifacts_dir / "plots_and_figures"
for d in (checkpoint_dir, recording_dir, model_dir, video_dir, figure_dir):
    d.mkdir(parents=True, exist_ok=True)

best_model_path = model_dir / "best_model.pt"
metrics_csv = figure_dir / "generation_metrics.csv"
results_path = artifacts_dir / "results.json"
print("run:", run_dir.name)

run: run-20261022_105305


# Environment

Next, we'll build the environment used to train Doomguy.

My Way Home is a maze of colored rooms. Doomguy can turn, walk forward, or step sideways. He starts in a random room, facing a random direction, and the green vest is always in the same room. Reaching it is worth +1. Every tic alive costs 0.0001, and the episode times out after 2100 tics. That's all there is: no extra rewards or penalties, and no early endings.

`make_env` turns each frame into an 84x84 grayscale image and stacks the last 4 frames so the network can see motion. The novelty bonus gets added later, inside the training loop, rather than in a wrapper, because it depends on the agent's own networks.

In [7]:
def make_env(config_file=None):
    # Stock maze. Pass a config file to load my_way_home.cfg directly.
    kwargs = dict(
        render_mode="rgb_array",
        frame_skip=FRAME_SKIP,
        screen_resolution=vzd.ScreenResolution.RES_200X125,
        max_buttons_pressed=MAX_BUTTONS_PRESSED,
        treat_episode_timeout_as_truncation=False,
    )
    if DOOM_WAD:
        kwargs["doom_game_path"] = DOOM_WAD
    env = VizdoomEnv(config_file=config_file, **kwargs) if config_file else gymnasium.make(ENV_NAME, **kwargs)

    h, w, _ = env.observation_space["screen"].shape
    env = TransformObservation(
        env,
        lambda obs: (obs["screen"] @ np.array([0.299, 0.587, 0.114])).astype(np.uint8),
        Box(low=0, high=255, shape=(h, w), dtype=np.uint8),
    )  # grayscale
    env = ResizeObservation(env, OBS_SIZE)  # 84x84
    env = FrameStackObservation(env, stack_size=STACK_SIZE)  # last 4 frames
    return env

Training, evaluation, and recording all use the same stock maze. We create `NUM_ENVS` copies, each in its own process, so they can step at the same time.

In [8]:
env = gymnasium.vector.AsyncVectorEnv(
    [make_env] * NUM_ENVS, autoreset_mode=gymnasium.vector.AutoresetMode.SAME_STEP
)

objc[82949]: Class SDLApplication is implemented in both /opt/miniconda3/lib/python3.13/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x12fb8ca68) and /opt/miniconda3/lib/python3.13/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x14a00c890). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[82949]: Class SDLAppDelegate is implemented in both /opt/miniconda3/lib/python3.13/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x12fb8cab8) and /opt/miniconda3/lib/python3.13/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x14a00c8e0). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[82949]: Class SDLTranslatorResponder is implemented in both /opt/miniconda3/lib/python3.13/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x12fb8cb30) and /opt/miniconda3/lib/python3.13/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x14a00c958). This may cause spurio

# Hyperparameters
PPO collects `LEARN_STEP` steps of fresh experience, 2048 in total, or 128 from each of the 16 environments. It then makes `UPDATE_EPOCHS` passes over that batch in chunks of `BATCH_SIZE` and throws the batch away. `CLIP_COEF` limits how far one update can move the policy away from the policy that collected the data, and `ENT_COEF` adds a small bonus for keeping the action probabilities spread out. These are the same starting values as the other PPO notebooks.

The reward settings follow the RND paper. The game reward and the novelty reward each get their own discount: `GAMMA_EXT` is 0.999, so a vest found far in the future still counts, and `GAMMA_INT` is 0.99, so novelty mostly pulls him toward what's close by. `EXT_COEF` and `INT_COEF` weight the two advantages when they're added together. The paper gives the game reward twice the weight, so once he's found the vest, that's what wins.

We train for about 6,000,000 steps. Every 25 updates, 51,200 steps, make up one generation: we play and record the evaluation episodes and save a checkpoint, which gives us 120 to compare.

The RND settings are explained with the RND module below.

In [9]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"  # Apple Silicon GPU
else:
    device = "cpu"
print("device:", device)

PPO_HP = {
    "BATCH_SIZE": 256,
    "LR": 2.5e-4,
    "GAMMA_EXT": 0.999,        # discount for the game reward
    "GAMMA_INT": 0.99,         # discount for the novelty reward
    "EXT_COEF": 2.0,           # weight of the game-reward advantage
    "INT_COEF": 1.0,           # weight of the novelty advantage; 0 turns RND off
    "GAE_LAMBDA": 0.95,        # how far ahead the advantage looks
    "LEARN_STEP": 2048,        # fresh steps across all envs before an update
    "UPDATE_EPOCHS": 4,        # passes over that batch
    "CLIP_COEF": 0.1,          # how far one update can move the policy
    "ENT_COEF": 0.01,          # bonus for keeping actions spread out
    "VF_COEF": 0.5,            # weight of the value loss
    "MAX_GRAD_NORM": 0.5,      # gradient clipping
    "MAX_STEPS": 6_144_000,
    "EVO_STEPS": 51_200,       # steps per generation, 25 updates
}

RND_HP = {
    "FEATURE_DIM": 512,        # size of the target's random embedding
    "UPDATE_PROPORTION": 1.0,  # share of each minibatch the predictor trains on
    "NORM_INIT_STEPS": 8192,   # random-action steps used to seed the frame statistics
}

NET_CONFIG = {
    "channels": (64, 128, 128),  # conv layers, double the Atari DQN widths
    "feature_dim": 512,          # encoder output
    "hidden_size": 512,          # hidden layer of the policy and value heads
}
OBS_SHAPE = env.single_observation_space.shape
N_ACTIONS = int(env.single_action_space.n)
FRAME_SHAPE = (1, *OBS_SIZE)  # RND looks at the newest frame only

device: mps


# Networks
## Encoder
The encoder follows the shape of the original Atari DQN: three convolutional layers, then a linear layer down to a feature embedding. It's twice as wide, though, with 64, 128, and 128 filters and a 512-dimensional embedding, since the maze has a lot of rooms to tell apart. It takes the raw uint8 frames and scales them to [0, 1] itself, so the rest of the notebook never has to remember to. The sizes all come from `NET_CONFIG`, and the size of the flattened conv output is measured with a dummy frame rather than hardcoded.

`layer_init` is the orthogonal initialization most PPO implementations use. It's one of those small details that makes a surprising difference to how quickly PPO gets going.

In [10]:
def layer_init(layer, gain=np.sqrt(2), bias=0.0):
    nn.init.orthogonal_(layer.weight, gain)
    nn.init.constant_(layer.bias, bias)
    return layer


def conv_stack(in_channels, channels, activation=nn.ReLU):
    c1, c2, c3 = channels
    return nn.Sequential(
        layer_init(nn.Conv2d(in_channels, c1, kernel_size=8, stride=4)), activation(),
        layer_init(nn.Conv2d(c1, c2, kernel_size=4, stride=2)), activation(),
        layer_init(nn.Conv2d(c2, c3, kernel_size=3, stride=1)), activation(),
        nn.Flatten(),
    )


def conv_out(conv, input_shape):
    with torch.no_grad():
        return conv(torch.zeros(1, *input_shape)).shape[1]


class Encoder(nn.Module):
    def __init__(self, obs_shape, channels, feature_dim):
        super().__init__()
        self.conv = conv_stack(obs_shape[0], channels)
        self.linear = nn.Sequential(layer_init(nn.Linear(conv_out(self.conv, obs_shape), feature_dim)), nn.ReLU())

    def forward(self, obs):
        return self.linear(self.conv(obs.float() / 255.0))

## Actor-critic
`ActorCritic` puts two heads on one shared encoder. The policy head turns the embedding into a probability for each action, and the value head estimates how well things are going so the actor can tell whether an action beat expectations.

The value head has two outputs, one for the game's reward and one for the novelty reward, as in the RND paper. Keeping them apart lets each reward be discounted under its own rules, which is what makes non-episodic novelty possible (see the rollout buffer below). Column 0 is extrinsic and column 1 is intrinsic.

The policy's output layer starts with tiny weights so every action starts out about equally likely, and the value head starts at unit scale. Those gains also come from the standard PPO recipe.

In [11]:
class ActorCritic(nn.Module):
    def __init__(self, obs_shape, n_actions, channels, feature_dim, hidden_size):
        super().__init__()
        self.encoder = Encoder(obs_shape, channels, feature_dim)
        self.policy_head = nn.Sequential(
            layer_init(nn.Linear(feature_dim, hidden_size)), nn.ReLU(),
            layer_init(nn.Linear(hidden_size, n_actions), gain=0.01),
        )
        self.value_head = nn.Sequential(
            layer_init(nn.Linear(feature_dim, hidden_size)), nn.ReLU(),
            layer_init(nn.Linear(hidden_size, 2), gain=1.0),  # (extrinsic, intrinsic)
        )

    def forward(self, obs):
        # Returns (action distribution, values) for a batch of raw observations; values are (batch, 2).
        features = self.encoder(obs)
        return Categorical(logits=self.policy_head(features)), self.value_head(features)

    @torch.no_grad()
    def act(self, obs):
        # Sample actions for a batch of raw observations; returns (actions, log_probs, values).
        dist, values = self(obs)
        actions = dist.sample()
        return actions, dist.log_prob(actions), values

## Random Network Distillation
RND has two networks that look at the newest frame on its own, separately from the policy.

- The **target** is a randomly initialized conv net that maps a frame to a 512-number embedding. It's frozen, so its embedding of a frame never changes. It's an arbitrary but fixed fingerprint of what's on screen.
- The **predictor** has the same conv layers plus a deeper head, and it's trained to reproduce the target's embedding. Its error on a frame is the novelty: low on frames like ones it's trained on, high on new ones.

The target's weights can't be learned, so the predictor can't game the error by making the target easy. The deeper predictor head is from the paper, and it gives the predictor enough capacity to eventually match the target everywhere it has been. The conv layers use the Atari sizes with leaky ReLUs, like the paper.

`RunningMeanStd` keeps a running mean and variance. RND uses two of them:

- **Frame statistics.** Before a frame goes into either network, each pixel is shifted by its running mean, divided by its running standard deviation, and clipped to ±5. Without this, a random target's output barely changes from one frame to the next, and there's nothing to learn. The statistics are seeded with `NORM_INIT_STEPS` steps of random actions before training starts, then updated after every rollout.
- **Reward scale.** The raw error has no natural scale, so it's divided by the running standard deviation of the discounted sum of novelty rewards. `RewardForwardFilter` keeps that discounted sum for each environment. This keeps the novelty reward on a steady scale as the predictor improves, which is what lets `INT_COEF` mean the same thing all through training.

In [12]:
class RunningMeanStd:
    def __init__(self, shape=()):
        self.mean = np.zeros(shape, dtype=np.float64)
        self.var = np.ones(shape, dtype=np.float64)
        self.count = 1e-4

    def update(self, x):
        # Merge a batch shaped (n, *shape) into the running statistics.
        batch_mean, batch_var, n = x.mean(axis=0), x.var(axis=0), len(x)
        delta = batch_mean - self.mean
        total = self.count + n
        self.mean = self.mean + delta * n / total
        self.var = (self.var * self.count + batch_var * n + delta**2 * self.count * n / total) / total
        self.count = total

    def state(self):
        return {"mean": self.mean, "var": self.var, "count": self.count}


class RewardForwardFilter:
    # Running discounted sum of rewards per environment, the quantity the reward scale is measured on.
    def __init__(self, gamma):
        self.gamma, self.total = gamma, None

    def update(self, rewards):
        self.total = rewards if self.total is None else self.total * self.gamma + rewards
        return self.total


class RND(nn.Module):
    def __init__(self, frame_shape, feature_dim):
        super().__init__()
        channels = (32, 64, 64)
        self.target = nn.Sequential(
            conv_stack(frame_shape[0], channels, nn.LeakyReLU),
            layer_init(nn.Linear(conv_out(conv_stack(frame_shape[0], channels), frame_shape), feature_dim)),
        )
        self.predictor = nn.Sequential(
            conv_stack(frame_shape[0], channels, nn.LeakyReLU),
            layer_init(nn.Linear(conv_out(conv_stack(frame_shape[0], channels), frame_shape), 512)), nn.ReLU(),
            layer_init(nn.Linear(512, 512)), nn.ReLU(),
            layer_init(nn.Linear(512, feature_dim)),
        )
        for p in self.target.parameters():
            p.requires_grad_(False)

    def forward(self, frames):
        # Per-frame novelty for a batch of normalized frames: the predictor's mean squared error.
        return (self.predictor(frames) - self.target(frames)).pow(2).mean(dim=-1)


def normalize_frames(frames, frame_rms):
    # uint8 frames shaped (batch, 1, 84, 84) on `device`, normalized per pixel and clipped.
    mean = torch.as_tensor(frame_rms.mean, dtype=torch.float32, device=device)
    std = torch.as_tensor(np.sqrt(frame_rms.var), dtype=torch.float32, device=device)
    return ((frames.float() - mean) / (std + 1e-8)).clamp(-5, 5)

# PPO
## Rollout buffer
`RolloutBuffer` holds one batch of experience as arrays shaped `(steps, envs, ...)`. Observations stay as uint8 on the CPU and only move to the GPU a minibatch at a time.

It also stores `next_frames`, the newest frame of the observation each action led to. That's the frame RND scores, so the novelty bonus is paid for arriving somewhere new.

`gae` is standard generalized advantage estimation. `dones[t]` means step `t` ended the episode, so the bootstrap from the next value is cut right there. `compute_advantages` runs it twice, once per value head, each with its own discount, and adds the two advantages together, weighted by `EXT_COEF` and `INT_COEF`, for the policy update.

The game reward is episodic: reaching the vest or timing out ends the episode, and nothing after it counts. Novelty is non-episodic, as in the paper: its GAE ignores episode ends, so the stream just carries on into the next episode. That way ending an episode never costs him future novelty, so he has no reason to avoid the vest, and no reason to stall to keep an episode going.

In [13]:
def gae(rewards, values, last_value, dones, gamma, gae_lambda):
    advantages = np.zeros_like(values)
    running = np.zeros_like(last_value)
    for t in reversed(range(len(rewards))):
        next_value = last_value if t == len(rewards) - 1 else values[t + 1]
        not_done = 1.0 - dones[t]
        delta = rewards[t] + gamma * next_value * not_done - values[t]
        running = delta + gamma * gae_lambda * not_done * running
        advantages[t] = running
    return advantages


class RolloutBuffer:
    def __init__(self, n_steps, num_envs, obs_shape, frame_shape):
        shape = (n_steps, num_envs)
        self.obs = np.zeros((*shape, *obs_shape), dtype=np.uint8)
        self.next_frames = np.zeros((*shape, *frame_shape), dtype=np.uint8)
        self.actions = np.zeros(shape, dtype=np.int64)
        self.log_probs = np.zeros(shape, dtype=np.float32)
        self.values = np.zeros((*shape, 2), dtype=np.float32)  # (extrinsic, intrinsic)
        self.rewards = np.zeros(shape, dtype=np.float32)
        self.dones = np.zeros(shape, dtype=np.float32)
        self.n_steps = n_steps

    def add(self, t, obs, next_frames, actions, log_probs, values, rewards, dones):
        self.obs[t], self.next_frames[t], self.actions[t] = obs, next_frames, actions
        self.log_probs[t], self.values[t] = log_probs, values
        self.rewards[t], self.dones[t] = rewards, dones

    def compute_advantages(self, intrinsic_rewards, last_values):
        lam = PPO_HP["GAE_LAMBDA"]
        extrinsic = gae(self.rewards, self.values[..., 0], last_values[:, 0], self.dones, PPO_HP["GAMMA_EXT"], lam)
        intrinsic = gae(intrinsic_rewards, self.values[..., 1], last_values[:, 1], np.zeros_like(self.dones),
                        PPO_HP["GAMMA_INT"], lam)
        self.advantages = PPO_HP["EXT_COEF"] * extrinsic + PPO_HP["INT_COEF"] * intrinsic
        self.returns = np.stack([extrinsic, intrinsic], axis=-1) + self.values

    def minibatches(self, batch_size):
        # Yields shuffled minibatches as dicts of tensors on `device`, covering the buffer once.
        flat = lambda x: x.reshape(-1, *x.shape[2:])
        data = {k: flat(getattr(self, k)) for k in
                ("obs", "next_frames", "actions", "log_probs", "values", "advantages", "returns")}
        n = len(data["actions"])
        for idx in np.array_split(np.random.permutation(n), n // batch_size):
            yield {k: torch.as_tensor(v[idx], device=device) for k, v in data.items()}

## Collecting a rollout
`collect_rollout` steps all 16 environments `LEARN_STEP / NUM_ENVS` times and fills the buffer. It returns both values of the observation the rollout stopped on, which GAE uses to bootstrap, along with the score of each episode that finished.

The vector env resets a finished environment in the same step, so `next_obs` already shows the new episode for that slot. The real last frame is in `info["final_obs"]`, and we swap it in, so RND scores the frame he actually reached rather than a random new room.

In [14]:
def to_tensor(obs):
    return torch.as_tensor(obs, device=device)


def collect_rollout(model, env, buffer, obs, scores):
    finished_scores = []
    for t in range(buffer.n_steps):
        actions, log_probs, values = model.act(to_tensor(obs))
        actions = actions.cpu().numpy()
        next_obs, reward, terminated, truncated, info = env.step(actions)
        done = terminated | truncated

        real_next_obs = next_obs.copy()
        for i in np.flatnonzero(info.get("_final_obs", [])):
            real_next_obs[i] = info["final_obs"][i]
        buffer.add(t, obs, real_next_obs[:, -1:], actions, log_probs.cpu().numpy(), values.cpu().numpy(), reward, done)

        scores += reward
        for i in np.flatnonzero(done):
            finished_scores.append(scores[i])
            scores[i] = 0.0
        obs = next_obs

    last_values = model.act(to_tensor(obs))[2].cpu().numpy()
    return obs, last_values, finished_scores

## Novelty reward
`novelty` scores every frame in the rollout with the current predictor, once per rollout, so every transition in the batch is scored by the same model. `intrinsic_rewards` then runs each environment's errors through `RewardForwardFilter`, folds those discounted sums into the reward statistics, and divides the errors by their running standard deviation. After that, the frame statistics are updated with the rollout's frames, ready for the next one.

In [15]:
@torch.no_grad()
def novelty(rnd, frames, frame_rms, chunk=1024):
    errors = []
    for i in range(0, len(frames), chunk):
        errors.append(rnd(normalize_frames(to_tensor(frames[i:i + chunk]), frame_rms)).cpu().numpy())
    return np.concatenate(errors)


def intrinsic_rewards(rnd, buffer, frame_rms, reward_rms, reward_filter):
    # Returns (normalized rewards shaped (steps, envs), raw errors) and updates both running statistics.
    frames = buffer.next_frames.reshape(-1, *FRAME_SHAPE)
    errors = novelty(rnd, frames, frame_rms).reshape(buffer.actions.shape)
    discounted = np.stack([reward_filter.update(e) for e in errors])
    reward_rms.update(discounted.reshape(-1))
    frame_rms.update(frames.astype(np.float64))
    return (errors / np.sqrt(reward_rms.var)).astype(np.float32), errors

## The update
`ppo_update` is PPO's clipped update with the RND predictor loss added on. The predictor only sees normalized frames, and it doesn't share any weights with the policy, so adding the two losses together just lets one optimizer step both. `UPDATE_PROPORTION` lets the predictor train on a random share of each minibatch. The paper used 25% with 128 environments, to keep the predictor from catching up too fast. With 16 environments, it trains on all of them.

The pieces of the PPO loss:

- **Policy loss**: the probability ratio between the new and old policy, times the advantage, clipped to `1 ± CLIP_COEF` so one update can't move the policy too far.
- **Value loss**: squared error against the GAE returns, with the same clipping applied to how far the value can move. It's summed over both value heads.
- **Entropy**: subtracted, so spreading the probabilities out lowers the loss.

Advantages are normalized per minibatch.

In [16]:
def ppo_update(model, rnd, optimizer, buffer, frame_rms):
    clip = PPO_HP["CLIP_COEF"]
    params = [p for group in optimizer.param_groups for p in group["params"]]
    totals, n_minibatches = {}, 0
    for _ in range(PPO_HP["UPDATE_EPOCHS"]):
        for mb in buffer.minibatches(PPO_HP["BATCH_SIZE"]):
            dist, values = model(mb["obs"])
            log_probs = dist.log_prob(mb["actions"])
            entropy = dist.entropy().mean()

            advantages = (mb["advantages"] - mb["advantages"].mean()) / (mb["advantages"].std() + 1e-8)
            ratio = torch.exp(log_probs - mb["log_probs"])
            policy_loss = torch.max(-advantages * ratio, -advantages * torch.clamp(ratio, 1 - clip, 1 + clip)).mean()

            clipped_values = mb["values"] + torch.clamp(values - mb["values"], -clip, clip)
            value_loss = 0.5 * torch.max((values - mb["returns"]) ** 2, (clipped_values - mb["returns"]) ** 2).sum(-1).mean()

            errors = rnd(normalize_frames(mb["next_frames"], frame_rms))
            mask = (torch.rand(len(errors), device=device) < RND_HP["UPDATE_PROPORTION"]).float()
            rnd_loss = (errors * mask).sum() / mask.sum().clamp(min=1.0)

            loss = policy_loss + PPO_HP["VF_COEF"] * value_loss - PPO_HP["ENT_COEF"] * entropy + rnd_loss
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(params, PPO_HP["MAX_GRAD_NORM"])
            optimizer.step()

            step_metrics = {
                "loss": loss, "policy_loss": policy_loss, "value_loss": value_loss,
                "entropy": entropy, "rnd_loss": rnd_loss,
            }
            for k, v in step_metrics.items():
                totals[k] = totals.get(k, 0.0) + v.item()
            n_minibatches += 1
    return {k: v / n_minibatches for k, v in totals.items()}

## Evaluation and recording
Evaluation happens at the end of every generation, on its own environment so the 16 training environments never get interrupted mid-episode. The policy plays `EVAL_EPISODES` episodes with no novelty bonus, so the score is the real game reward. Like the other PPO notebooks, it keeps sampling from the policy instead of always taking the most likely action, which can leave him stuck turning in place.

Every one of those episodes is recorded as a ViZDoom `.lmp` demo while it plays. A demo stores only the inputs, a few KB per episode, and gets replayed at full resolution later. The Gymnasium wrapper doesn't expose demo recording, so `reset_with_lmp` temporarily replaces the reset function with one that starts a recording. `flush_lmp` begins the next episode so the file is actually written.

One gotcha: ViZDoom hangs if the demo path is longer than about 112 characters, and the run folder on Colab with Drive is longer than that. So each demo is written under a short temporary name in the working directory and moved into the run's `recordings` folder once the generation's episodes are done.

In [17]:
def reset_with_lmp(env, lmp_path):
    base = env.unwrapped
    orig_reset = base.reset

    def recorded_reset(*, seed=None, options=None):
        gymnasium.Env.reset(base, seed=seed)
        game_seed = int(
            base.np_random.integers(0, np.iinfo(np.uint32).max + 1, dtype=np.uint32)
        )
        base.game.set_seed(game_seed)
        if not base.game.is_running():
            base.game.init()
        # ViZDoom hangs if the demo path is longer than ~112 characters, so pass it relative to the working directory
        base.game.new_episode(os.path.relpath(lmp_path))
        base.state = base.game.get_state()
        return base._VizdoomEnv__collect_observations(), {}

    base.reset = recorded_reset
    try:
        return env.reset()
    finally:
        base.reset = orig_reset


def flush_lmp(env):
    env.unwrapped.game.new_episode()

In [18]:
def play(env, model, obs):
    # Run one episode from `obs`; returns (total_reward, actions).
    total, actions, done = 0.0, [], False
    while not done:
        action = int(model.act(to_tensor(obs[None]))[0].item())
        obs, reward, terminated, truncated, _ = env.step(action)
        total += reward
        actions.append(action)
        done = terminated or truncated
    return total, actions


def evaluate_and_record(model, env, generation):
    # Play EVAL_EPISODES recorded episodes; returns (rewards, demo paths in recording_dir).
    rewards, temp_demos = [], []
    for episode in range(EVAL_EPISODES):
        temp_demo = Path(f"_eval_demo_{episode}.lmp")  # short and relative, for the path length limit
        obs, _ = reset_with_lmp(env, temp_demo)
        reward, _ = play(env, model, obs)
        rewards.append(reward)
        temp_demos.append(temp_demo)
    flush_lmp(env)

    demos = []
    for episode, temp_demo in enumerate(temp_demos):
        demo = recording_dir / f"gen_{generation:03d}_episode_{episode}.lmp"
        shutil.move(temp_demo, demo)
        demos.append(demo)
    return rewards, demos

A checkpoint is a single file holding the actor-critic, both RND networks, the optimizer, both running statistics, and the settings needed to rebuild the networks. `load_model` only rebuilds the actor-critic, since that's all it takes to play.

In [19]:
def save_checkpoint(path, model, rnd, optimizer, frame_rms, reward_rms, steps):
    torch.save({
        "steps": steps,
        "config": {"obs_shape": OBS_SHAPE, "n_actions": N_ACTIONS, **NET_CONFIG},
        "model": model.state_dict(),
        "rnd": rnd.state_dict(),
        "optimizer": optimizer.state_dict(),
        "frame_rms": frame_rms.state(),
        "reward_rms": reward_rms.state(),
        "ppo_hp": PPO_HP,
        "rnd_hp": RND_HP,
    }, path)


def load_model(path):
    checkpoint = torch.load(path, map_location=device, weights_only=False)
    model = ActorCritic(**checkpoint["config"]).to(device)
    model.load_state_dict(checkpoint["model"])
    return model.eval()

# Training
Now we build everything. One optimizer covers the actor-critic and the RND predictor; the target never trains. The `eps` of 1e-5 is another PPO convention.

Before training starts, the frame statistics are seeded by playing `NORM_INIT_STEPS` steps of random actions across all the environments. Those steps don't count toward training. The environments are reset afterwards so every training episode starts clean.

In [20]:
model = ActorCritic(OBS_SHAPE, N_ACTIONS, **NET_CONFIG).to(device)
rnd = RND(FRAME_SHAPE, RND_HP["FEATURE_DIM"]).to(device)
optimizer = torch.optim.Adam([*model.parameters(), *rnd.predictor.parameters()], lr=PPO_HP["LR"], eps=1e-5)
buffer = RolloutBuffer(PPO_HP["LEARN_STEP"] // NUM_ENVS, NUM_ENVS, OBS_SHAPE, FRAME_SHAPE)
eval_env = make_env(scenario_cfg)
print(f"actor-critic: {sum(p.numel() for p in model.parameters()):,} params, "
      f"RND predictor: {sum(p.numel() for p in rnd.predictor.parameters()):,} params")

frame_rms = RunningMeanStd(FRAME_SHAPE)
reward_rms = RunningMeanStd()
reward_filter = RewardForwardFilter(PPO_HP["GAMMA_INT"])

env.reset()
for _ in tqdm(range(RND_HP["NORM_INIT_STEPS"] // NUM_ENVS), desc="frame statistics"):
    next_obs, *_ = env.step(env.action_space.sample())
    frame_rms.update(next_obs[:, -1:].astype(np.float64))

actor-critic: 4,036,424 params, RND predictor: 2,203,296 params


frame statistics: 100%|██████████| 512/512 [00:02<00:00, 218.91it/s]


Each update has four steps: collect a rollout, score it with RND, compute advantages for the game reward and the novelty reward, and run the PPO and predictor losses together. Each generation runs 25 updates, then plays and records its evaluation episodes and saves a checkpoint.

Each generation logs one row: the PPO losses, the predictor loss, the average novelty reward before and after scaling, the mean score of the training episodes that finished, the share of them that reached the vest, and the mean and standard deviation of the evaluation episodes. The evaluation rewards and demo paths for every generation are kept in `eval_rewards` and `eval_demos`.

This one is a long run. A good time to step away from the machine.

In [21]:
history, eval_rewards, eval_demos = [], [], []
updates_per_generation = PPO_HP["EVO_STEPS"] // PPO_HP["LEARN_STEP"]
total_steps = 0

obs, _ = env.reset()
scores = np.zeros(NUM_ENVS)
pbar = tqdm(total=PPO_HP["MAX_STEPS"], unit="step")
for generation in range(PPO_HP["MAX_STEPS"] // PPO_HP["EVO_STEPS"]):
    model.train()
    update_metrics, episode_scores, intrinsic_means, novelty_means = [], [], [], []
    for _ in range(updates_per_generation):
        obs, last_values, finished = collect_rollout(model, env, buffer, obs, scores)
        episode_scores += finished

        rewards_int, errors = intrinsic_rewards(rnd, buffer, frame_rms, reward_rms, reward_filter)
        intrinsic_means.append(rewards_int.mean())
        novelty_means.append(errors.mean())

        buffer.compute_advantages(rewards_int, last_values)
        update_metrics.append(ppo_update(model, rnd, optimizer, buffer, frame_rms))
        total_steps += PPO_HP["LEARN_STEP"]
        pbar.update(PPO_HP["LEARN_STEP"])

    model.eval()
    rewards, demos = evaluate_and_record(model, eval_env, generation)
    eval_rewards.append(rewards)
    eval_demos.append(demos)

    row = {"steps": total_steps, **pd.DataFrame(update_metrics).mean().to_dict()}
    row["intrinsic_reward"] = float(np.mean(intrinsic_means))
    row["raw_novelty"] = float(np.mean(novelty_means))
    row["train_score"] = float(np.mean(episode_scores)) if episode_scores else np.nan
    row["train_episodes"] = len(episode_scores)
    row["vest_rate"] = float(np.mean([s > 0 for s in episode_scores])) if episode_scores else np.nan
    row["eval_mean"], row["eval_std"] = float(np.mean(rewards)), float(np.std(rewards))
    history.append(row)
    pbar.set_postfix(eval=f"{row['eval_mean']:.2f}", vest=f"{row['vest_rate']:.0%}", novelty=f"{row['raw_novelty']:.3f}")

    save_checkpoint(checkpoint_dir / f"ckpt_{total_steps}.pt", model, rnd, optimizer, frame_rms, reward_rms, total_steps)
pbar.close()
env.close()
eval_env.close()

 59%|█████▉    | 3620864/6144000 [3:40:22<3:34:52, 195.70step/s, eval=-0.21, novelty=0.104, vest=1%] 

KeyboardInterrupt: 

 59%|█████▉    | 3620864/6144000 [3:40:38<3:34:52, 195.70step/s, eval=-0.21, novelty=0.104, vest=1%]

# Best Generation
Every generation was already scored during training, so picking the best one is just a lookup. The generation with the highest mean evaluation reward wins; ties go to the later generation. Its checkpoint is copied to `artifacts`.

In [ ]:
best_gen = max(range(len(history)), key=lambda g: (history[g]["eval_mean"], g))
best_checkpoint = checkpoint_dir / f"ckpt_{history[best_gen]['steps']}.pt"
shutil.copy2(best_checkpoint, best_model_path)
print(f"best: gen {best_gen} ({best_checkpoint.name}), mean eval reward {history[best_gen]['eval_mean']:.2f}, "
      f"rewards={np.round(eval_rewards[best_gen], 2).tolist()}")

# Analysis
Let's see how it did! Training is noisy and the eval episodes aren't seeded, so your curves won't match these exactly. The notes below are about the general shape.

## Metrics
One row per generation: the PPO losses and entropy, the predictor loss, the novelty reward, the training score and vest rate, and the evaluation mean and standard deviation. The full table gets saved as a CSV in `artifacts`. Here we just show the first few generations, the best one, and the last few.

In [ ]:
metrics = pd.DataFrame(history)
metrics.index.name = "generation"
shown = pd.concat([metrics.head(3), metrics.loc[[best_gen]], metrics.tail(3)])
shown[~shown.index.duplicated()]

In [ ]:
metrics.to_csv(metrics_csv)

## Charts
Saved to `artifacts/plots_and_figures`, and shown here too.

In [ ]:
def save(fig, name):
    fig.savefig(figure_dir / name, dpi=150, bbox_inches="tight")
    plt.show()


gens = metrics.index

**Loss.** PPO's losses don't say much about how well the agent plays. The value loss is how far off the critic is, and with novelty in the reward the critic is chasing a moving target, so expect it to stay noisy. The policy loss stays near zero by design, since clipping keeps every update small.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(gens, metrics["value_loss"], marker="o")
ax1.set(xlabel="Generation", ylabel="Mean value loss", title="Critic (value) loss per generation")
ax2.plot(gens, metrics["policy_loss"], marker="o")
ax2.set(xlabel="Generation", ylabel="Mean policy loss", title="Actor (policy) loss per generation")
save(fig, "loss.png")

**RND.** The left chart is the predictor's loss, which is the raw novelty averaged over the frames he saw. It should drop quickly at first as the predictor learns the starting rooms, then level off or bump back up whenever he reaches new parts of the maze. The right chart is the novelty reward actually paid per step, after scaling. If it slides toward zero and stays there, he's stopped finding anything new.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(gens, metrics["rnd_loss"], marker="o")
ax1.set(xlabel="Generation", ylabel="Mean predictor loss", title="RND predictor loss per generation")
ax2.plot(gens, metrics["intrinsic_reward"], marker="o")
ax2.set(xlabel="Generation", ylabel="Mean reward per step", title="Novelty reward per generation")
save(fig, "rnd.png")

**Entropy.** Entropy measures how spread out the action probabilities are. It starts near the maximum, the dashed line where every action is equally likely, and should fall as the agent gets more confident. If it crashes toward zero early, the agent stopped exploring too soon.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(gens, metrics["entropy"], marker="o")
ax.axhline(np.log(N_ACTIONS), color="gray", linestyle="--", label="Maximum (uniform policy)")
ax.set(xlabel="Generation", ylabel="Mean entropy", title="Policy entropy per generation")
ax.legend()
save(fig, "entropy.png")

**Average reward.** A found vest lands near 1. A timeout lands near -0.2, because every tic costs a tiny bit and the episode is long. Blue shows the recorded evaluation episodes at the end of each generation, with the shaded band representing one standard deviation. Orange shows the training episodes that finished during the generation, which use the same stock rewards.

The band stays wide while he's still wandering, since one vest and one timeout pull the average in opposite directions. If blue sits near 1, he found the vest.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
(line,) = ax.plot(gens, metrics["eval_mean"], marker="o", label=f"Evaluation ({EVAL_EPISODES} episodes, {SCENARIO_NAME})")
ax.fill_between(gens, metrics["eval_mean"] - metrics["eval_std"],
                metrics["eval_mean"] + metrics["eval_std"], color=line.get_color(), alpha=0.2)
ax.plot(gens, metrics["train_score"], marker="o", label="Training episodes")
ax.axvline(best_gen, color="gray", linestyle="--", label=f"Best generation ({best_gen})")
ax.set(xlabel="Generation", ylabel="Average reward", title="Average reward per generation")
ax.legend()
save(fig, "average_reward.png")

**Vest rate.** This is the share of each generation's training episodes that reached the vest. It's the steadiest view of progress, since a generation finishes far more training episodes than evaluation ones. A random policy finds the vest in only a few percent of episodes, mostly when it starts close by.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(gens, metrics["vest_rate"], marker="o")
ax.axvline(best_gen, color="gray", linestyle="--", label=f"Best generation ({best_gen})")
ax.set(xlabel="Generation", ylabel="Share of training episodes", title="Training episodes that reached the vest")
ax.set_ylim(0, 1)
ax.legend()
save(fig, "vest_rate.png")

# Recording
The demos were all recorded during training, so all that's left is turning them into videos.

The first `RECORD_EPISODES` demos from the best generation are copied into `artifacts/video` as `best_episode_*.lmp`. With `RECORD_CHECKPOINTS` set to `"all"`, the same number from every other generation are converted in place in the run folder. Existing best-model videos are cleared first so they don't get mixed with the current run.

In [ ]:
assert RECORD_CHECKPOINTS in ("best", "all")
for stale in video_dir.glob("best_episode_*"):
    stale.unlink()

lmps, lmp_rewards = [], {}
for gen in range(len(history)) if RECORD_CHECKPOINTS == "all" else [best_gen]:
    for episode, demo in enumerate(eval_demos[gen][:RECORD_EPISODES]):
        if gen == best_gen:
            demo = Path(shutil.copy2(demo, video_dir / f"best_episode_{episode}.lmp"))
        lmps.append(demo)
        lmp_rewards[demo] = eval_rewards[gen][episode]
        print(f"gen {gen:>3}  episode {episode}  reward={eval_rewards[gen][episode]:.2f}  -> {demo.name}")

`lmp_to_mp4` replays each demo at 720p with the HUD enabled. The result looks much better than the 84x84 input used for training because we're replaying the actions in a fresh game rather than saving what the network saw. CRF 18, tuned for Doom's flat textures, keeps the files small enough for GitHub. Audio capture is silent on my Mac, so locally generated videos have no sound. Encoding takes a while because the encoder is configured to prioritize compression quality over speed.

In [ ]:
SAMPLE_RATE = 22050
VIDEO_PARAMS = dict(  # x264 at constant quality: CRF 18 is visually lossless, and "animation" suits Doom's flat textures
    fps=35, codec="libx264", quality=None, macro_block_size=8,
    ffmpeg_params=["-crf", "18", "-preset", "veryslow", "-tune", "animation", "-movflags", "+faststart"],
)


def lmp_to_mp4(lmp_path):
    """Replay an .lmp at 720p with HUD and audio, and write it next to the .lmp as .mp4."""
    lmp_path = Path(lmp_path)
    mp4_path = lmp_path.with_suffix(".mp4")
    silent_path = lmp_path.with_name(lmp_path.stem + "_silent.mp4")
    wav_path = lmp_path.with_suffix(".wav")

    game = vzd.DoomGame()
    game.load_config(scenario_cfg)
    if DOOM_WAD:
        game.set_doom_game_path(DOOM_WAD)
    game.set_screen_resolution(vzd.ScreenResolution.RES_1280X720)
    game.set_render_hud(True)
    game.set_render_corpses(True)
    game.set_window_visible(False)
    game.set_audio_buffer_enabled(True)
    game.set_audio_sampling_rate(vzd.SamplingRate.SR_22050)
    game.set_audio_buffer_size(1)
    game.add_game_args("+snd_efx 0")
    game.set_mode(vzd.Mode.SPECTATOR)
    game.init()

    short_lmp = Path("_replay_demo.lmp")  # short and relative, for the path length limit
    shutil.copy2(lmp_path, short_lmp)
    frames, audio = [], []
    try:
        game.replay_episode(str(short_lmp))
        while not game.is_episode_finished():
            state = game.get_state()
            if state is not None:
                frames.append(np.transpose(state.screen_buffer, (1, 2, 0)))
                if state.audio_buffer is not None:
                    audio.append(state.audio_buffer)
            game.advance_action()
    finally:
        game.close()
        short_lmp.unlink()

    imageio.mimsave(silent_path, frames, **VIDEO_PARAMS)

    audio = np.concatenate(audio) if audio else np.zeros(1)
    if not audio.any():  # no sound captured (e.g. on macOS): keep the silent video
        silent_path.replace(mp4_path)
        return mp4_path

    wavfile.write(wav_path, SAMPLE_RATE, audio)
    subprocess.run(
        [imageio_ffmpeg.get_ffmpeg_exe(), "-y", "-i", silent_path, "-i", wav_path,
         "-c:v", "copy", "-c:a", "aac", "-shortest", mp4_path],
        check=True, capture_output=True,
    )
    silent_path.unlink()
    wav_path.unlink()
    return mp4_path

In [ ]:
mp4s = [lmp_to_mp4(lmp) for lmp in lmps]
mp4s

For a quick preview, the highest-scoring episode is converted to a deliberately small GIF that plays directly in the notebook and on GitHub. The full-resolution MP4s remain in `artifacts/video`.

In [ ]:
GIF_WIDTH = 240
GIF_FPS = 10


def mp4_to_gif(mp4_path):
    """Write a small GIF next to an MP4, with a palette built from the video so the colors hold up."""
    gif_path = Path(mp4_path).with_suffix(".gif")
    vf = (
        f"fps={GIF_FPS},scale={GIF_WIDTH}:-1:flags=lanczos,split[a][b];"
        "[a]palettegen=max_colors=128:stats_mode=diff[p];[b][p]paletteuse=dither=bayer:bayer_scale=5:diff_mode=rectangle"
    )
    subprocess.run([imageio_ffmpeg.get_ffmpeg_exe(), "-y", "-i", mp4_path, "-vf", vf, gif_path], check=True, capture_output=True)
    return gif_path


best_mp4 = max(
    (m for m in mp4s if m.name.startswith("best")),
    key=lambda m: lmp_rewards[m.with_suffix(".lmp")],
)
best_gif = mp4_to_gif(best_mp4)
print(f"{best_gif.name}  reward={lmp_rewards[best_mp4.with_suffix('.lmp')]:.2f}  ({best_gif.stat().st_size / 2**20:.1f} MB)")
display(Image(filename=str(best_gif)))

# Results
A short summary of the run, also saved to `artifacts/results.json` so runs are easy to compare later.

In [ ]:
results = {
    "run": run_dir.name,
    "scenario": SCENARIO_NAME,
    "iwad": IWAD_NAME,
    "best_checkpoint": best_checkpoint.name,
    "best_generation": best_gen,
    "best_steps": int(metrics.loc[best_gen, "steps"]),
    "generations": len(history),
    "eval_mean": round(float(metrics.loc[best_gen, "eval_mean"]), 4),
    "eval_std": round(float(metrics.loc[best_gen, "eval_std"]), 4),
    "eval_rewards": [round(float(r), 4) for r in eval_rewards[best_gen]],
    "best_video": [str(m.relative_to(experiment_dir)) for m in mp4s if m.name.startswith("best")],
    "best_gif": str(best_gif.relative_to(experiment_dir)),
    "net_config": NET_CONFIG,
    "ppo_hp": PPO_HP,
    "rnd_hp": RND_HP,
}
results_path.write_text(json.dumps(results, indent=2))
pd.Series(results, name="value").to_frame()

# Wrapping Up
My Way Home is where plain exploration struggles. Doomguy has to recognize the rooms and walk to the vest from a random start, and the only reward shows up when he gets there. RND gives him something to chase in the meantime: every unfamiliar room pays out until the predictor has learned it.

The interesting comparison is against the same notebook with `INT_COEF = 0`, which is plain PPO with everything else held fixed. If the RND run finds the vest and the baseline doesn't, RND did its job. If it doesn't, the RND chart is the first place to look: a predictor loss that drops to nothing in the first few generations means the bonus ran out before he got anywhere new.

Happy Coding!